# Stage 2 Rollout Analysis: Cycle-Normalized View

This notebook analyzes `rl_output/stage2_eval_rollout.npz` with cycle-normalized plots rather than raw phase-scatter plots.

Scope and caveats:
- `phase_proxy = arctan2(sin_phi, cos_phi)` is reconstructed for reference, but it is **not** treated as a trusted full gait phase.
- Gait cycles are segmented directly from repeated motion signals within each episode.
- This is a frozen-walker bootstrap experiment with an effort proxy, not a full human-exo co-adaptation study.
- The effort signal is useful for controller debugging, but it should not be overinterpreted as a clinical or physiological outcome.
- Cycle duration is reported in samples because the rollout does not include a physical `dt`.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.signal import correlate, find_peaks
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

rollout_name = "stage2_eval_rollout.npz"
rollout_path = None  # Optional override, e.g. Path("/absolute/path/to/stage2_eval_rollout.npz")

candidate_paths = [
    Path("rl_output") / rollout_name,
    Path("rl") / "rl_output" / rollout_name,
    Path("..") / "rl_output" / rollout_name,
    Path("..") / ".." / "rl_output" / rollout_name,
]

if rollout_path is not None:
    rollout_path = Path(rollout_path).expanduser().resolve()

if rollout_path is None:
    for candidate in candidate_paths:
        if candidate.exists():
            rollout_path = candidate.resolve()
            break

if rollout_path is None:
    matches = sorted(Path.cwd().rglob(rollout_name))
    if matches:
        rollout_path = matches[0].resolve()

if rollout_path is None:
    raise FileNotFoundError(
        "Could not find rl_output/stage2_eval_rollout.npz. Set rollout_path manually in this cell if needed."
    )

baseline_effort = None
log_path = None
for candidate in [
    rollout_path.parent / "exo_training_log.json",
    Path("rl_output/exo_training_log.json"),
    Path("rl/rl_output/exo_training_log.json"),
    Path("../rl_output/exo_training_log.json"),
]:
    if candidate.exists():
        log_path = candidate.resolve()
        with open(log_path) as f:
            baseline_effort = json.load(f).get("baseline_effort")
        break

rollout_path

In [ ]:
with np.load(rollout_path) as data:
    raw = {k: np.asarray(data[k]).reshape(-1) for k in data.files}

required = [
    "episode",
    "step",
    "reward",
    "effort",
    "tau_r",
    "tau_l",
    "hip_r",
    "hip_l",
    "hipd_r",
    "hipd_l",
    "pelvis_vx",
    "torso_pitch",
    "sin_phi",
    "cos_phi",
]
missing = [k for k in required if k not in raw]
if missing:
    raise KeyError(f"Missing keys in rollout: {missing}")

df = pd.DataFrame(raw)
df["sample"] = np.arange(len(df))
df["phase_proxy"] = np.arctan2(df["sin_phi"], df["cos_phi"])
df["phase_proxy_pct"] = np.mod(df["phase_proxy"], 2 * np.pi) / (2 * np.pi) * 100

overview = pd.Series(
    {
        "samples": len(df),
        "episodes": int(df["episode"].nunique()),
        "mean reward": float(df["reward"].mean()),
        "mean effort": float(df["effort"].mean()),
        "baseline effort": np.nan if baseline_effort is None else float(baseline_effort),
    }
)
display(overview.to_frame("value"))
df.head()

## Cycle Detection Method

Primary rule:
- Cycles are defined from consecutive repeated extrema of `hip_r` within each episode.

Why this choice:
- `hip_r` is one of the clearest oscillatory signals in the rollout.
- Using repeated extrema is easy to inspect and avoids over-trusting the rough `phase_proxy`.
- Samples before the first detected boundary and after the last detected boundary in each episode are dropped, so only complete cycles are analyzed.

Robustness fallback:
- If prominent `hip_r` maxima are too sparse, the notebook tries a more relaxed `hip_r` extrema rule and then `hipd_r` zero-crossings.
- This makes the segmentation less brittle on short or weakly oscillatory rollouts.
- Even with these fallbacks, the result is still only a repeatable segmentation heuristic, not a true biomechanical gait-event detector.


In [ ]:
norm_points = 100
norm_grid = np.linspace(0, 100, norm_points)
signals = ["tau_r", "tau_l", "effort", "hip_r", "hip_l"]

df["cycle_id"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
df["cycle_pct"] = np.nan

boundary_rows = []
cycle_rows = []
norm_frames = []
detection_rows = []
next_cycle_id = 0

def estimate_period_guess(y):
    if len(y) < 8:
        return max(4, len(y) - 1)
    y_centered = y - y.mean()
    ac = correlate(y_centered, y_centered, mode="full")
    ac = ac[len(y) - 1 :]
    min_lag = max(3, len(y) // 20)
    max_lag = max(min_lag + 1, len(y) // 2)
    if max_lag <= min_lag:
        return max(6, len(y) // 3)
    window = ac[min_lag:max_lag]
    if len(window) == 0 or np.allclose(window, 0):
        return max(6, len(y) // 3)
    return int(min_lag + np.argmax(window))

def enforce_min_gap(indices, min_gap):
    indices = np.asarray(indices, dtype=int)
    if len(indices) <= 1:
        return indices
    kept = [int(indices[0])]
    for idx in indices[1:]:
        if idx - kept[-1] >= min_gap:
            kept.append(int(idx))
    return np.asarray(kept, dtype=int)

def choose_boundaries(y, yd, period_guess):
    amp = np.ptp(y)
    sigma = np.std(y)
    min_gap = max(3, int(0.25 * period_guess))

    candidates = []

    for method, signal, prominence, distance in [
        ("hip_r peaks", y, max(0.06 * amp, 0.08 * sigma, 1e-6), max(3, int(0.35 * period_guess))),
        ("hip_r peaks (relaxed)", y, max(0.03 * amp, 0.04 * sigma, 1e-6), max(2, int(0.20 * period_guess))),
        ("hip_r minima", -y, max(0.06 * amp, 0.08 * sigma, 1e-6), max(3, int(0.35 * period_guess))),
    ]:
        idx, _ = find_peaks(signal, distance=distance, prominence=prominence)
        candidates.append((method, enforce_min_gap(idx, min_gap)))

    zero_crossings = [
        ("hipd_r +to- zero crossings", np.where((yd[:-1] > 0) & (yd[1:] <= 0))[0] + 1),
        ("hipd_r -to+ zero crossings", np.where((yd[:-1] < 0) & (yd[1:] >= 0))[0] + 1),
    ]
    for method, idx in zero_crossings:
        candidates.append((method, enforce_min_gap(idx, min_gap)))

    candidates = [(method, idx) for method, idx in candidates if len(idx) >= 2]
    if not candidates:
        return "none", np.array([], dtype=int)

    def candidate_score(idx):
        if len(idx) < 2:
            return (-1, -np.inf)
        diffs = np.diff(idx)
        regularity = 0.0 if len(diffs) < 2 else np.std(diffs) / max(np.mean(diffs), 1e-9)
        return (len(idx), -regularity)

    method, boundaries = max(candidates, key=lambda item: candidate_score(item[1]))
    return method, np.asarray(boundaries, dtype=int)

for episode_id, ep_df in df.groupby("episode", sort=True):
    y = ep_df["hip_r"].to_numpy()
    yd = ep_df["hipd_r"].to_numpy()
    ep_index = ep_df.index.to_numpy()

    period_guess = estimate_period_guess(y)
    method, boundaries = choose_boundaries(y, yd, period_guess)

    raw_lengths = np.diff(boundaries) if len(boundaries) >= 2 else np.array([], dtype=int)
    if len(raw_lengths) >= 3:
        median_length = float(np.median(raw_lengths))
        boundary_pairs = [
            (a, b)
            for a, b in zip(boundaries[:-1], boundaries[1:])
            if 0.35 * median_length <= (b - a) <= 1.65 * median_length
        ]
        if not boundary_pairs:
            boundary_pairs = list(zip(boundaries[:-1], boundaries[1:]))
    else:
        boundary_pairs = list(zip(boundaries[:-1], boundaries[1:]))

    episode_cycle_count = 0
    for start_idx, end_idx in boundary_pairs:
        segment = ep_df.iloc[start_idx : end_idx + 1].copy()
        n_samples = len(segment)
        if n_samples < 5:
            continue

        cycle_rows.append(
            {
                "cycle_id": next_cycle_id,
                "episode": int(episode_id),
                "start_sample": int(segment["sample"].iloc[0]),
                "end_sample": int(segment["sample"].iloc[-1]),
                "n_samples": int(n_samples),
                "boundary_method": method,
            }
        )

        assign_index = segment.index[:-1]
        if len(assign_index) > 0:
            df.loc[assign_index, "cycle_id"] = next_cycle_id
            df.loc[assign_index, "cycle_pct"] = np.linspace(0, 100, len(assign_index), endpoint=False)

        x_old = np.linspace(0, 100, n_samples)
        norm_frame = pd.DataFrame(
            {"cycle_id": next_cycle_id, "episode": int(episode_id), "cycle_pct": norm_grid}
        )
        for name in signals:
            norm_frame[name] = np.interp(norm_grid, x_old, segment[name].to_numpy())
        norm_frames.append(norm_frame)

        next_cycle_id += 1
        episode_cycle_count += 1

    detection_rows.append(
        {
            "episode": int(episode_id),
            "episode_samples": int(len(ep_df)),
            "period_guess_samples": int(period_guess),
            "boundary_method": method,
            "n_boundaries": int(len(boundaries)),
            "n_complete_cycles": int(episode_cycle_count),
        }
    )

    boundary_rows.extend(
        {
            "episode": int(episode_id),
            "sample": int(ep_df["sample"].iloc[idx]),
            "index": int(ep_index[idx]),
            "hip_r": float(y[idx]),
            "boundary_method": method,
        }
        for idx in boundaries
    )

boundary_df = pd.DataFrame(boundary_rows)
cycle_table = pd.DataFrame(cycle_rows)
detection_table = pd.DataFrame(detection_rows)
has_cycles = not cycle_table.empty

if has_cycles:
    norm_df = pd.concat(norm_frames, ignore_index=True)
    cycle_overview = pd.Series(
        {
            "detected cycles": int(len(cycle_table)),
            "samples kept in complete cycles": int(df["cycle_id"].notna().sum()),
            "mean cycle length (samples)": float(cycle_table["n_samples"].mean()),
            "std cycle length (samples)": float(cycle_table["n_samples"].std(ddof=0)),
        }
    )
else:
    norm_df = pd.DataFrame(columns=["cycle_id", "episode", "cycle_pct", *signals])
    cycle_overview = pd.Series(
        {
            "detected cycles": 0,
            "samples kept in complete cycles": 0,
            "mean cycle length (samples)": np.nan,
            "std cycle length (samples)": np.nan,
        }
    )

display(cycle_overview.to_frame("value"))
display(detection_table)
cycle_table.head()

In [ ]:
episode_breaks = np.flatnonzero(np.diff(df["episode"].to_numpy())) + 1

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df["sample"], df["hip_r"], color="tab:blue", lw=1.5, label="hip_r")
if not boundary_df.empty:
    ax.scatter(
        boundary_df["sample"],
        boundary_df["hip_r"],
        s=24,
        color="tab:red",
        zorder=3,
        label="detected boundaries",
    )
if has_cycles:
    for x in cycle_table["start_sample"]:
        ax.axvline(x, color="tab:red", alpha=0.08, lw=1)
for x in episode_breaks:
    ax.axvline(x, color="k", alpha=0.20, lw=1, linestyle="--")
ax.set_title("Cycle boundary detection from repeated hip motion")
ax.set_xlabel("sample")
ax.set_ylabel("hip_r")
ax.legend()
plt.tight_layout()
plt.show()

## Interpretation

- The table above shows what boundary rule succeeded for each episode and how many complete cycles were recovered.
- If `n_complete_cycles` is zero across most episodes, the rollout is probably too short, too weakly oscillatory, or too irregular for reliable cycle segmentation.
- That is a data/rollout limitation as much as a detector-setting issue.


In [ ]:
if not has_cycles:
    display(Markdown(
        "## Cycle Overlays Skipped\n\n"
        "No complete cycles were detected. This usually means the rollout episodes ended before two repeatable boundaries appeared, or the hip motion was too weak/noisy for reliable segmentation."
    ))
else:
    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

    for _, cycle in norm_df.groupby("cycle_id"):
        axes[0].plot(cycle["cycle_pct"], cycle["tau_r"], color="tab:blue", alpha=0.20, lw=1)
        axes[1].plot(cycle["cycle_pct"], cycle["tau_l"], color="tab:orange", alpha=0.20, lw=1)
        axes[2].plot(cycle["cycle_pct"], cycle["effort"], color="tab:green", alpha=0.20, lw=1)

    axes[0].set_title("tau_r across normalized cycles")
    axes[0].set_ylabel("tau_r")
    axes[1].set_title("tau_l across normalized cycles")
    axes[1].set_ylabel("tau_l")
    axes[2].set_title("effort across normalized cycles")
    axes[2].set_ylabel("effort")
    axes[2].set_xlabel("cycle (%)")

    plt.tight_layout()
    plt.show()

## Interpretation

- Tight bundles suggest that the controller output repeats from cycle to cycle.
- Wide spread or drift suggests variable torque timing or inconsistent walker dynamics.
- The effort overlay should be read as a stability/debug view of the proxy, not as proof of biomechanical benefit.


In [ ]:
if not has_cycles:
    mean_df = pd.DataFrame(columns=signals)
    std_df = pd.DataFrame(columns=signals)
    display(Markdown(
        "## Mean ± Std Skipped\n\n"
        "Cycle-normalized mean ± std plots need at least one complete detected cycle."
    ))
else:
    mean_df = norm_df.groupby("cycle_pct")[signals].mean()
    std_df = norm_df.groupby("cycle_pct")[signals].std().fillna(0.0)

    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

    for name, color, label in [("tau_r", "tab:blue", "tau_r"), ("tau_l", "tab:orange", "tau_l")]:
        axes[0].plot(mean_df.index, mean_df[name], color=color, lw=2, label=label)
        axes[0].fill_between(
            mean_df.index,
            mean_df[name] - std_df[name],
            mean_df[name] + std_df[name],
            color=color,
            alpha=0.20,
        )
    axes[0].set_title("Mean ± std torque across normalized cycle")
    axes[0].set_ylabel("torque")
    axes[0].legend()

    axes[1].plot(mean_df.index, mean_df["effort"], color="tab:green", lw=2)
    axes[1].fill_between(
        mean_df.index,
        mean_df["effort"] - std_df["effort"],
        mean_df["effort"] + std_df["effort"],
        color="tab:green",
        alpha=0.20,
    )
    axes[1].set_title("Mean ± std effort across normalized cycle")
    axes[1].set_ylabel("effort")

    for name, color, label in [("hip_r", "tab:blue", "hip_r"), ("hip_l", "tab:orange", "hip_l")]:
        axes[2].plot(mean_df.index, mean_df[name], color=color, lw=2, label=label)
        axes[2].fill_between(
            mean_df.index,
            mean_df[name] - std_df[name],
            mean_df[name] + std_df[name],
            color=color,
            alpha=0.20,
        )
    axes[2].set_title("Mean ± std hip angle across normalized cycle")
    axes[2].set_ylabel("hip angle")
    axes[2].set_xlabel("cycle (%)")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## Interpretation

- Mean ± std emphasizes repeatable structure while keeping cycle-to-cycle variability visible.
- Use the torque bands to judge how periodic and consistent the controller is.
- Use the hip traces mainly as context for where the controller acts within the repeated motion, not as a full biomechanical gait analysis.


In [ ]:
def mean_cycle_correlation(cycle_matrix):
    arr = cycle_matrix.to_numpy()
    if arr.size == 0:
        return np.nan
    template = arr.mean(axis=0)
    corrs = []
    for row in arr:
        if np.std(row) == 0 or np.std(template) == 0:
            corrs.append(np.nan)
        else:
            corrs.append(np.corrcoef(row, template)[0, 1])
    return float(np.nanmean(corrs))

delta_tau_r_parts = [np.diff(g["tau_r"].to_numpy()) for _, g in df.groupby("episode") if len(g) > 1]
delta_tau_l_parts = [np.diff(g["tau_l"].to_numpy()) for _, g in df.groupby("episode") if len(g) > 1]
delta_tau_r = np.concatenate(delta_tau_r_parts) if delta_tau_r_parts else np.array([np.nan])
delta_tau_l = np.concatenate(delta_tau_l_parts) if delta_tau_l_parts else np.array([np.nan])

tau_r_consistency = np.nan
tau_l_consistency = np.nan
effort_consistency = np.nan
lag_pct = np.nan
best_corr = np.nan
lr_relation = "undetermined because no complete cycles were segmented"

if has_cycles:
    tau_r_cycles = norm_df.pivot(index="cycle_id", columns="cycle_pct", values="tau_r").sort_index()
    tau_l_cycles = norm_df.pivot(index="cycle_id", columns="cycle_pct", values="tau_l").sort_index()
    effort_cycles = norm_df.pivot(index="cycle_id", columns="cycle_pct", values="effort").sort_index()

    tau_r_consistency = mean_cycle_correlation(tau_r_cycles)
    tau_l_consistency = mean_cycle_correlation(tau_l_cycles)
    effort_consistency = mean_cycle_correlation(effort_cycles)

    mean_tau_r = mean_df["tau_r"].to_numpy()
    mean_tau_l = mean_df["tau_l"].to_numpy()
    a = mean_tau_r - mean_tau_r.mean()
    corrs = []
    for shift in range(len(mean_tau_l)):
        b = np.roll(mean_tau_l, shift) - mean_tau_l.mean()
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        corrs.append(np.dot(a, b) / denom if denom > 0 else np.nan)
    corrs = np.asarray(corrs, dtype=float)

    if not np.all(np.isnan(corrs)):
        best_shift = int(np.nanargmax(corrs))
        best_corr = float(np.nanmax(corrs))
        if best_shift > len(mean_tau_l) // 2:
            best_shift -= len(mean_tau_l)
        lag_pct = 100 * best_shift / len(mean_tau_l)

        if abs(lag_pct) <= 5:
            lr_relation = "mostly overlapping / synchronized"
        elif abs(lag_pct) <= 30:
            lr_relation = "phase-shifted"
        else:
            lr_relation = "strongly offset"

metrics = pd.Series(
    {
        "mean effort": float(df["effort"].mean()),
        "std effort": float(df["effort"].std(ddof=0)),
        "mean |tau_r|": float(np.abs(df["tau_r"]).mean()),
        "mean |tau_l|": float(np.abs(df["tau_l"]).mean()),
        "mean |Δtau_r|": float(np.nanmean(np.abs(delta_tau_r))),
        "mean |Δtau_l|": float(np.nanmean(np.abs(delta_tau_l))),
        "mean cycle length (samples)": np.nan if cycle_table.empty else float(cycle_table["n_samples"].mean()),
        "std cycle length (samples)": np.nan if cycle_table.empty else float(cycle_table["n_samples"].std(ddof=0)),
        "min cycle length (samples)": np.nan if cycle_table.empty else float(cycle_table["n_samples"].min()),
        "max cycle length (samples)": np.nan if cycle_table.empty else float(cycle_table["n_samples"].max()),
        "tau_r cycle consistency": tau_r_consistency,
        "tau_l cycle consistency": tau_l_consistency,
        "effort cycle consistency": effort_consistency,
        "left-right lag (% cycle)": lag_pct,
        "left-right max circular corr": best_corr,
    }
)

if baseline_effort is not None and baseline_effort > 0:
    metrics.loc["baseline effort"] = float(baseline_effort)
    metrics.loc["effort reduction vs baseline (%)"] = float(
        100 * (baseline_effort - df["effort"].mean()) / baseline_effort
    )

display(metrics.to_frame("value"))
print("Left-right relationship")
if has_cycles and np.isfinite(lag_pct):
    print(f"- Best circular lag: {lag_pct:+.1f}% of the normalized cycle")
    print(f"- Max circular correlation: {best_corr:.3f}")
    print(f"- Interpretation: {lr_relation}")
    print("- Positive lag means the left mean torque must be shifted later in the normalized cycle to best align with the right mean torque.")
else:
    print("- Undetermined from cycle-normalized analysis because no complete cycles were segmented.")

In [ ]:
effort_cv = float(df["effort"].std(ddof=0) / max(abs(df["effort"].mean()), 1e-9))

if has_cycles:
    torque_periodicity = np.nanmean([tau_r_consistency, tau_l_consistency])
    if torque_periodicity >= 0.80:
        periodic_text = (
            f"Yes. The torques look clearly periodic across detected cycles (mean cycle-correlation {torque_periodicity:.2f})."
        )
    elif torque_periodicity >= 0.60:
        periodic_text = (
            f"Mostly yes. The torques show a repeated pattern, but with noticeable cycle-to-cycle variability (mean cycle-correlation {torque_periodicity:.2f})."
        )
    else:
        periodic_text = (
            f"Only weakly. The torques do not collapse tightly across detected cycles (mean cycle-correlation {torque_periodicity:.2f})."
        )
else:
    periodic_text = (
        "Undetermined from cycle-normalized analysis because no complete cycles were segmented. "
        "This usually means the rollout is too short per episode or does not show repeatable boundaries clearly enough."
    )

if has_cycles and np.isfinite(lag_pct):
    if abs(lag_pct) <= 5:
        lr_text = (
            f"Left and right torques are mostly overlapping / synchronized, with a best circular lag of {lag_pct:+.1f}% of the normalized cycle."
        )
    elif abs(lag_pct) <= 30:
        lr_text = (
            f"Left and right torques are phase-shifted by about {abs(lag_pct):.1f}% of the normalized cycle."
        )
    else:
        lr_text = (
            f"Left and right torques are strongly offset, with a best circular lag of about {abs(lag_pct):.1f}% of the normalized cycle."
        )
else:
    lr_text = "Undetermined from cycle-normalized analysis because no complete cycles were segmented."

if has_cycles and np.isfinite(effort_consistency):
    if effort_consistency >= 0.80 and effort_cv < 0.20:
        effort_stability = "The effort proxy looks fairly stable across detected cycles."
    elif effort_consistency >= 0.60 and effort_cv < 0.35:
        effort_stability = "The effort proxy looks moderately stable, with some cycle-to-cycle variation."
    else:
        effort_stability = "The effort proxy varies noticeably across cycles."
else:
    if effort_cv < 0.20:
        effort_stability = "From the raw rollout alone, the effort proxy looks fairly stable."
    elif effort_cv < 0.35:
        effort_stability = "From the raw rollout alone, the effort proxy looks moderately stable."
    else:
        effort_stability = "From the raw rollout alone, the effort proxy varies noticeably."

if baseline_effort is None or baseline_effort <= 0:
    effort_change = "Reduction versus baseline cannot be determined from the rollout alone because no baseline reference is embedded in the `.npz`."
else:
    reduction_pct = 100 * (baseline_effort - df["effort"].mean()) / baseline_effort
    if reduction_pct > 5:
        effort_change = f"Relative to the saved training-log baseline, mean effort is lower by about {reduction_pct:.1f}%."
    elif reduction_pct < -5:
        effort_change = f"Relative to the saved training-log baseline, mean effort is higher by about {-reduction_pct:.1f}%."
    else:
        effort_change = f"Relative to the saved training-log baseline, mean effort is roughly unchanged ({reduction_pct:.1f}%)."

limit_lines = [
    "Cycle boundaries come from repeated `hip_r` extrema and `hipd_r` zero-crossing fallbacks, not true gait events such as heel strike.",
    "The reconstructed `phase_proxy` is only a rough internal state proxy and is not used here as a trusted full gait phase.",
    "This is a frozen-walker bootstrap experiment using an effort proxy, not a full human-exo co-adaptation result.",
    "Cycle timing is reported in samples because the rollout does not include a physical time step.",
]
if not has_cycles:
    limit_lines.append("No complete cycles were segmented in this rollout, so cycle-normalized conclusions are limited.")
if has_cycles and len(cycle_table) < 3:
    limit_lines.append(f"Only {len(cycle_table)} complete cycles were detected, so the cycle-averaged traces are based on a small sample.")

summary_md = "## Final Summary\n\n"
summary_md += f"- **Are the torques clearly periodic?** {periodic_text}\n"
summary_md += f"- **Are left and right torques phase-shifted or mostly overlapping?** {lr_text}\n"
summary_md += f"- **Does the effort proxy appear reduced and stable?** {effort_change} {effort_stability}\n"
summary_md += "- **What are the limitations of this analysis?**\n"
for line in limit_lines:
    summary_md += f"  - {line}\n"

display(Markdown(summary_md))